# 文本预处理：NLP 的第一步
> 难度标签：高级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：08_自然语言处理 | 源文件：文本预处理.py | 核心功能：分词、去停用词、词形还原、jieba 中文分词

## 概述

文本预处理是 NLP 的基础。原始文本需要经过分词、清洗、标准化等步骤才能喂给模型。中文分词尤其特殊——没有天然的空格分隔。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import re
from collections import Counter

## 数学原理

### 1. 文本预处理的流程

文本预处理是 NLP 管道的第一步，将原始文本转换为可被模型处理的规范化格式：

$$\text{原始文本} \xrightarrow{\text{清洗}} \xrightarrow{\text{分词}} \xrightarrow{\text{归一化}} \xrightarrow{\text{去停用词}} \text{词序列}$$

### 2. 中文分词

中文没有空格分隔词，需要显式分词。设输入字符序列为 $S = (c_1, c_2, \ldots, c_n)$，分词的目标是找到最优切分：

$$W^* = \arg\max_{W} P(W) = \arg\max_{w_1, w_2, \ldots, w_m} \prod_{i=1}^{m} P(w_i)$$

**jieba** 使用的算法：
- 基于前缀词典构建 DAG（有向无环图）
- 用动态规划求解最大概率路径
- 未登录词用 HMM（隐马尔可夫模型）识别

### 3. 英文归一化

**小写化**：统一大小写，减少词汇表

$$\text{"The"} \to \text{"the"}, \quad \text{"APPLE"} \to \text{"apple"}$$

**去标点**：用正则表达式移除特殊字符

$$\text{clean}(s) = \text{replace}(s, [^a-zA-Z0-9\s], "")$$

**词干提取**（Stemming）：将词还原为词干

$$\text{"running"} \to \text{"run"}, \quad \text{"better"} \to \text{"good"}$$

常用 Porter Stemmer 基于规则的后缀剥离。

**词形还原**（Lemmatization）：基于词典的规范化，比词干提取更准确。

### 4. 停用词过滤

停用词（stopwords）是高频但低信息量的词（如"的"、"是"、"the"、"is"）。

设停用词集合为 $\mathcal{S}$，过滤后的词序列：

$$W' = (w \in W : w \notin \mathcal{S})$$

停用词的 IDF 极低，移除它们可以：
- 降低特征维度
- 减少噪声
- 提升下游模型效果

### 5. 词频分布与 Zipf 定律

自然语言中词频服从 **Zipf 定律**：

$$f(r) \propto \frac{1}{r^\alpha}, \quad \alpha \approx 1$$

其中 $r$ 是词的频率排名。这意味着：
- 少数词出现极多次（"的"、"是"）
- 大多数词只出现几次
- 词汇表大小随语料增长但增速递减

### 6. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `jieba.cut(text)` | 基于 DAG + DP 的最优切分 $W^*$ |
| `jieba.add_word()` | 向前缀词典添加自定义词 |
| `re.sub(r"[^a-zA-Z0-9\s]", "", text)` | 正则清洗：$S \to S'$ |
| `text.lower()` | 归一化：统一大小写 |
| `Counter(words)` | 统计词频 $c(w, d)$ |
| `stopwords` 集合 | 停用词集合 $\mathcal{S}$ |
| `PorterStemmer().stem()` | 词干提取 $w \to \text{stem}(w)$ |

### 1. 示例文本

运行 1. 示例文本 的代码，观察算法在该环节的行为。

In [ ]:
texts = [
    "机器学习是人工智能的一个重要分支,深度学习是机器学习的子领域。",
    "自然语言处理（NLP）是AI领域中最热门的研究方向之一。",
    "Python是最好的编程语言,没有之一！",
    "The quick brown fox jumps over the lazy dog.",
# --- 继续 ---
    "I am learning natural language processing with Python.",
]
print("=== 原始文本 ===")
for i, t in enumerate(texts):
    print(f"  {i+1}. {t}")

### 2. 中文分词

运行 2. 中文分词 的代码，观察算法在该环节的行为。

In [ ]:
# 使用 jieba 分词（如未安装则用简单示例）
try:
    import jieba
    HAS_JIEBA = True
except ImportError:
    HAS_JIEBA = False
# --- 输出结果 ---
    print("[SKIP] jieba 未安装，跳过本示例")
    import sys; sys.exit(0)
    HAS_JIEBA = False
    print("\njieba 未安装,使用简单分词示例")

if HAS_JIEBA:
    print("\n=== jieba 中文分词 ===")
    for text in texts[:3]:
        words = list(jieba.cut(text))
        print(f"  原文: {text}")
# --- 输出结果 ---
        print(f"  分词: {'|'.join(words)}")

    # 添加自定义词典
    jieba.add_word("自然语言处理")
    jieba.add_word("深度学习")
    print("\n添加自定义词后:")
    words = list(jieba.cut("自然语言处理是深度学习的应用"))
    print(f"  分词: {'|'.join(words)}")

### 3. 英文分词 + 小写化

运行 3. 英文分词 + 小写化 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 英文预处理 ===")
en_text = "The Quick Brown FOX jumps over the lazy dog. It's a beautiful day!"
# 小写化
lower = en_text.lower()
# 去除标点和特殊字符
clean = re.sub(r"[^a-zA-Z0-9\s]", "", lower)
# 分词
words = clean.split()
print(f"  原文: {en_text}")
print(f"  清洗后: {' '.join(words)}")

### 4. 去停用词

运行 4. 去停用词 的代码，观察算法在该环节的行为。

In [ ]:
# 中文停用词（简化版）
zh_stopwords = {"的", "是", "了", "在", "和", "有", "我", "不", "这", "就",
                "也", "都", "而", "及", "与", "或", "一个", "没有", "我们",
                "人们", "把", "那", "你", "他", "它", "她", "被", "让", "到"}
# 英文停用词
en_stopwords = {"the", "a", "an", "is", "are", "was", "were", "it", "its",
                "i", "you", "he", "she", "we", "they", "am", "to", "of",
                "in", "on", "at", "for", "with", "over", "about"}

if HAS_JIEBA:
    print("\n=== 去停用词（中文）===")
    text = "机器学习是人工智能的一个重要分支"
    words = list(jieba.cut(text))
    filtered = [w for w in words if w not in zh_stopwords and len(w.strip()) > 0]
# --- 输出结果 ---
    print(f"  原文: {text}")
    print(f"  分词: {words}")
    print(f"  去停用词: {filtered}")

print("\n=== 去停用词（英文）===")
words = clean.split()
filtered = [w for w in words if w not in en_stopwords]
print(f"  原文: {words}")
print(f"  去停用词: {filtered}")

### 5. 词干提取 / 词形还原

运行 5. 词干提取 / 词形还原 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 词干提取 / 词形还原 ===")
try:
    from nltk.stem import PorterStemmer, WordNetLemmatizer
    import nltk
    nltk.download("wordnet", quiet=True)
# --- 继续 ---
    nltk.download("omw-1.4", quiet=True)

    stemmer = PorterStemmer()
    lemmatizer = WordNetLemmatizer()

    words_en = ["running", "runs", "ran", "easily", "fairly", "studies", "studying"]
    print(f"  原词:     {words_en}")
    print(f"  词干提取: {[stemmer.stem(w) for w in words_en]}")
    print(f"  词形还原: {[lemmatizer.lemmatize(w) for w in words_en]}")
except ImportError:
# --- 输出结果 ---
    print("  NLTK 未安装,跳过词干提取演示")

### 6. 正则清洗

运行 6. 正则清洗 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 常用正则清洗模式 ===")
text = "用户 ID:12345 的邮箱 test@email.com,电话 138-0000-1234,金额 ¥100.50"
# 去除邮箱
no_email = re.sub(r"\S+@\S+", "[邮箱]", text)
print(f"  去邮箱: {no_email}")
# 去除电话
no_phone = re.sub(r"\d{3}[-.]?\d{4}[-.]?\d{4}", "[电话]", no_email)
print(f"  去电话: {no_phone}")
# 去除数字
no_num = re.sub(r"\d+", "[数字]", text)
print(f"  去数字: {no_num}")

### 7. 文本长度统计

运行 7. 文本长度统计 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 文本长度统计 ===")
for text in texts:
    char_len = len(text)
    word_len = len(text.split())
    print(f"  字符数={char_len:>3}, 词数={word_len:>2}: {text[:30]}...")

print("\n=== 文本预处理流程总结 ===")
print("1. 小写化/统一大小写")
print("2. 去除标点、特殊字符、HTML标签")
print("3. 分词（中文用 jieba/jieba-fast,英文用空格分割）")
print("4. 去停用词（高频无意义词）")
# --- 输出结果 ---
print("5. 词干提取/词形还原（统一词形）")
print("6. 去除过短的词（长度<2）")

## 关键代码解释

```python
import jieba
words = jieba.cut("我喜欢机器学习")
# 中文分词 → ["我", "喜欢", "机器学习"]
```

## 注意事项

1. 中文分词的歧义问题（"南京市长江大桥"）
2. 停用词表需要根据任务定制
3. 分词质量直接影响下游模型效果

## 总结与延伸

以上代码展示了 **文本预处理** 的完整流程。

**解读要点**：
- 观察 **词汇表大小** 和 **向量维度** 是否合理
- 检查相似词查询结果是否符合语义直觉
- 关注分类任务中的 **TF-IDF 权重** 分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **BPE/WordPiece**：子词分词，现代 NLP 的标准
- **SentencePiece**：语言无关的分词工具
- **中文 NLP 的特殊性**：分词、繁简转换、编码问题
